In [3]:
import pandas as pd
import os
import warnings
from pandasql import sqldf
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")
import glob
import openpyxl

### Nielsen

In [4]:
dir = os.getcwd()
# Important!! Make sure the file exist and refreshed first
sg_ldb = pd.read_excel(f'{dir}/../Data Source/Nielsen/Nielsen O+O May26_230626.xlsx', sheet_name='SG Nielsen Mass Medic')
sg_ldb.head()

,Periods,Markets,SECTOR,GLOBAL SEGMENT,LOCAL MANUF,LOCAL BRAND,CLTATTR1,Sales Value,Sales Units
0,Apr 23 - 4 w/e 30/04/23,Modern Trade/Singapore,DERMA,FEMALE,TOTAL OTHERS,NaN,NaN,422601.758,19557.0
1,Apr 23 - 4 w/e 30/04/23,Modern Trade/Singapore,DERMA,FEMALE,NaN,A-DERMA,ANTI AGING,1036.920,34.0
2,Apr 23 - 4 w/e 30/04/23,Modern Trade/Singapore,DERMA,FEMALE,NaN,A-DERMA,BASIC,1955.350,66.0
3,Apr 23 - 4 w/e 30/04/23,Modern Trade/Singapore,DERMA,FEMALE,NaN,A-DERMA,OIL CONTROL,9881.160,679.0
4,Apr 23 - 4 w/e 30/04/23,Modern Trade/Singapore,DERMA,FEMALE,NaN,ACNES,ANTI AGING,2112.430,79.0


In [5]:
def month_to_number(month):
    months = {'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'}
    return months.get(month, month)

In [6]:
# Important!! Brand Mapping - Any new Brands need to be added here
mass_medic = [
    'ACNE AID', 'ACNES', 'AVEENO', 'BALNEUM', 'BENZAC', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL',
    'DERMATIX', 'DERMAVEEN', 'DIFFERIN', 'DR.G', 'DR.YU', 'EGO', 'EUBOS', 'LACTACYD', 'LINOLA', 'MUSTELA',
    'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'FIRST AID BEAUTY',
    'AQUAPHOR', 'NOBACTER', 'LUBRIDERM', 'NEOSPORIN', 'DARROW', 'DEXERYL', 'ALERGIBON', 'ALPHYGIENE', 'BABIGOZ',
    'CANDERMYL', 'GALDERMA', 'GALDERMA OTHER', 'HELIOBLOC', 'HYDRODERM OMEGA', 'IOCON', 'IONIL', 'MACROLANE',
    'MICROBAN', 'MICROSUN', 'NESTLE', 'NUTRASPA', 'OBSERVANCE', 'PHYGIENE', 'R-GEN', 'SENTIAL', 'ACHE', 'ACNAID',
    'ACNE FREE', 'ACOFAR', 'ADDAX', 'AKILDIA', 'ALBOLENE', 'AMLACTIN', 'ANSEBIC', 'AQUA SOAP', 'AQUA-SOAP',
    'AVITIL', 'AZULENNE', 'BACCIDE', 'BEAUTY PLUS', 'BEDOOK', 'BEPANTHEN/BEPANTHOL', 'BETAGRANULOS', 'BIAFINE',
    'BIOBLAS', 'BIOCLIN', 'BIOLIQ', 'BIOXCIN', 'BLUE LIZARD', 'BODYSOL', 'BONAVEN', 'BOROLINE', 'CERAMOL',
    'CERTAIN DRI', 'CETOPIC', 'CHICCO', 'CICAMEL', 'COOPER', 'COTARYL', 'CRISTALIA', 'DECUBAL', 'DERMAC',
    'DERMACTIVE', 'DERMADRATE', 'DERMAGE', 'DERMAKERI', 'DERMENA', 'DERMON', 'DERSUPRIL', 'DEUMAVAN',
    'DOCTISSIMO PARAPHARMACIE', 'DR.LI', 'DR.LIDERMO', 'DRAYEX', 'DX2', 'E45', 'ELDOPAQUE', 'EMOLIENTA', 'EMOLIN',
    'EMOLIUM', 'EPIMAX', 'EVASOL', 'FARMOQUIMICA', 'FILTROSOL', 'FLUOCIN', 'FREI OEL (BOUHON)', 'GALENCO', 'GIFRER',
    'GILBERT', 'GOLD BOND', 'HAMILTON', 'HIDRAFIL', 'HIPOSOL', 'HYALIX', 'IDROVEL', 'IHADA', 'INFASIL',
    'INTERAPOTHEK', 'IRALTONE', 'ITANIDERM', 'KAMILODERM', 'KETOXIN', 'KINERASE', 'KORA', 'LACTIBON',
    'LACTO CALAMINE', 'LETI', 'LIFAR', 'LIPODERM', 'LOTRIMIN', 'MARQUE VERTE', 'MICRORET', 'MITOSYL', 'MODERM',
    'MULTIDERMOL', 'MUSSVITAL', 'NEUTRA LICE', 'NEUTRAPHARM', 'NORDIN', 'NUMIS', 'NUMIS MED', 'NURAPHARM',
    'NUTREM', 'NUTRISIL', 'OILATUM', 'OILLAN', 'OSMIN', 'OTC IBERICA', 'PANVEL DERMATIV', 'PARABOTICA',
    'PHARMACTIV', 'PHARMASEPT', 'PHISOHEX', 'PROCICAR', 'REGENERUM', 'RESTIV', 'RESTIVOIL', 'REVALESKIN', 'ROCHE',
    'ROGE CAVAILLES', 'ROYALCARE', 'RUGARD (SCHEFFLER)', 'SALILEX', 'SALLVE', 'SARNA', 'SAUGELLA', 'SEBORADIN',
    'SHADE', 'SMOOTH-E', 'SOLAR FOAM', 'S-OLE', 'SPECTRABAN', 'STANHOME FAMILY EXPERT', 'STIEFEL', 'STIEPROX',
    'STIPROX', 'STIPROXAL', 'TARMED', 'TRACTOPON', 'TRI DERMA MD', 'UREADERM', 'UVEIL-PS', 'UVESOL', 'VEA',
    'VENUSIA', 'VITA CITRAL', 'VITALIFE', 'ZODIAC','QV', 'BOBAI',
    'COLLAGE','DERMAREST','EPIZONE E','GLAMY LAB','LU MILD','NOLAVER','OXECURE','RIUP','SEBCUR','SELENGENA','SEROPIPE','SHAAN','STAR VILLE','STRONGVILLE','SYNOBAR','UREMOL',
    'URISEC','ZINPLEX'
]
df = pd.read_excel('Mapping.xlsx', sheet_name='Medic')
non_mass_medic = df.iloc[:, 0].dropna().astype(str).tolist()

# Add manually defined list
non_mass_medic1 = [
    'EUCERIN', 'LA ROCHE POSAY', 'VICHY', 'AVENE', 'DR MORITA', 'HIRUSCAR', 'URIAGE', 'SKINCEUTICALS', 'DECLEOR',
    'SANOFLORE', 'AQUAPHOR', 'BIODERMA', 'ISDIN', 'LIERAC', 'FILORGA', 'PROACTIV', 'ROC', 'WINONA', 'DR CILABO',
    'EMOLIUM', 'PHARMACERIS', 'CAUDALIE', 'NUXE', 'RODAN', 'FIELDS', 'LIBREDERM', 'MANTECORP', 'DR. CI : LABO'
]

# Combine both lists, remove duplicates, and standardize casing (if needed)
combined_non_mass_medic = list(set(non_mass_medic + non_mass_medic1))

# Optional: If you want to preserve order (Excel first, then manual additions)
combined_non_mass_medic = list(dict.fromkeys(non_mass_medic + non_mass_medic1))

print('Total mass_medic Brands: ', len(mass_medic))
print('Total non_mass_medic Brands: ', len(combined_non_mass_medic))

Total mass_medic Brands:  217
Total non_mass_medic Brands:  1133


In [7]:
def mass_medic_group(row):
    if row['LOCAL BRAND'] in mass_medic:
        return 'Mass Medical'
    else:
        return 'Non-Mass Medical'

In [8]:
# Important!! Try to understand the filter and logic here
sg_ldb['On/Offline'] = 'Offline'
sg_ldb['Year'] = sg_ldb['Periods'].str.extract('(\d+)', expand=False).astype(int) + 2000
sg_ldb['Month Name'] = sg_ldb['Periods'].str.split().str[0]
sg_ldb['Period'] = sg_ldb['Month Name'].apply(month_to_number)
sg_ldb['Platform'] = 'Nielsen'
sg_ldb['Category'] = 'SKINCARE'
sg_ldb['Sub-Category'] = sg_ldb['CLTATTR1']
sg_ldb['Mass/non-mass (subdivision)'] = sg_ldb.apply(mass_medic_group, axis=1)
sg_ldb['Brand'] = sg_ldb['LOCAL BRAND']
sg_ldb['Market'] = 'MEDIC MARKET'

In [9]:
# Filter and aggregate data for Brand and Market
ldb_brand = sg_ldb[sg_ldb['LOCAL BRAND'].isin(mass_medic) | sg_ldb['LOCAL BRAND'].isin(combined_non_mass_medic)].groupby(['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period'])[['Sales Value','Sales Units']].sum().reset_index()
ldb_market = sg_ldb.groupby(['On/Offline', 'Year', 'Platform', 'Category', 'Mass/non-mass (subdivision)', 'Market', 'Period'])[['Sales Value','Sales Units']].sum().reset_index()
ldb_market['Sub-Category'] = ''
ldb_market = ldb_market.rename(columns={'Market': 'Brand'})
ldb_market.head(3)

,On/Offline,Year,Platform,Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Sub-Category
0,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,04,733640.361,43516.0,
1,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,05,819606.816,49404.0,
2,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,06,968239.173,55875.0,


In [10]:
# Merge Brand and Market
sg_ldb_offline = pd.concat([ldb_market, ldb_brand], ignore_index=True)
sg_ldb_offline.head(3)

,On/Offline,Year,Platform,Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Sub-Category
0,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,04,733640.361,43516.0,
1,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,05,819606.816,49404.0,
2,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,06,968239.173,55875.0,


### OMT LDB

In [11]:
# Important!! Make sure the file exist and updated first
files = glob.glob(f'{dir}/../Data Source/OMT - O+O/SG LDB/*.xlsx')

# List to store DataFrames
dfs = []
# Read each file and append to the list
for file in files:
    df = pd.read_excel(file)
    dfs.append(df)

# Concatenate all DataFrames into one
ldb_data = pd.concat(dfs, ignore_index=True)

In [12]:
# Rename platform
def map_platform(row):
    if row['Mall Type'] == 'Shopee Mall' :
        return 'Shopee Mall'
    elif row['Mall Type'] == 'Lazada Mall' :
        return 'Lazada Mall'
    elif row['Mall Type'] == 'Tiktok Mall':
        return 'Tiktok Mall'
    else:
        return 'Others'

In [13]:
def map_division(row):
    if row['Brand'] in mass_medic:
        return 'Mass Medical'
    else:
        return 'Non-Mass Medical'

In [14]:
# Rename Columns for consistency
ldb_data = ldb_data.rename(columns={'Total Est. Sales Local': 'Sales Value', "Loreal 1P Est Sales Local": 'Loreal Sales Value', 'Total units sold': 'Sales Units', 'Category L2':'Category_x'})

In [15]:
# Important!! Try to understand the filter and logic here
ldb_data = ldb_data[ldb_data['Category L1'] == 'SKIN CARE']
# Keep only specified mall types
ldb_data = ldb_data[ldb_data['Mall Type'].isin(['Shopee Mall', 'Lazada Mall','Tiktok Mall'])]
# For SUN CARE, only include rows where Category L3 is 'FACE PROTECTION'
ldb_data = ldb_data[(ldb_data['Category_x'] != 'SUN CARE') | ((ldb_data['Category_x'] == 'SUN CARE') & (ldb_data['Category L3'] == 'FACE PROTECTION'))]
ldb_data['On/Offline'] = 'Online'
ldb_data[['Year', 'Period']] = ldb_data['Year Month'].str.split('-', expand=True)
ldb_data['Platform'] = ldb_data.apply(map_platform, axis=1)
ldb_data['Category'] = 'SKINCARE'
ldb_data['Sub-Category'] = ldb_data['Category_x']
ldb_data['Mass/non-mass (subdivision)'] = ldb_data.apply(map_division, axis=1)
ldb_data['Market'] = 'MEDIC MARKET'
ldb_data.info()

<class 'pandas.DataFrame'>
Index: 104067 entries, 0 to 122882
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Country                      104067 non-null  str    
 1   Year Month                   104067 non-null  str    
 2   Universe                     104067 non-null  str    
 3   Brand                        104067 non-null  str    
 4   Category L1                  104067 non-null  str    
 5   Category_x                   104067 non-null  str    
 6   Mall Type                    104067 non-null  str    
 7   Category L3                  104067 non-null  str    
 8   Product                      104067 non-null  str    
 9   Benefits                     104067 non-null  str    
 10  Formats                      104067 non-null  str    
 11  Sales Value                  104067 non-null  float64
 12  Loreal Sales Value           6698 non-null    float64
 13  Sales Units    

In [16]:
# Filter and aggregate data for Brand and Market
ldb_market = ldb_data.groupby(['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Market', 'Period'])[['Sales Value','Sales Units','Loreal Sales Value']].sum().reset_index()
ldb_market = ldb_market.rename(columns={'Market': 'Brand'})
ldb_brand = ldb_data[ldb_data['Brand'].isin(mass_medic) | ldb_data['Brand'].isin(combined_non_mass_medic)].groupby(['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period'])[['Sales Value','Sales Units','Loreal Sales Value']].sum().reset_index()
ldb_brand = ldb_brand[~((ldb_brand['Sub-Category'] == 'Body Care') & (ldb_brand['Brand'] != 'SKINCEUTICALS'))]
ldb_brand.tail(3)

,On/Offline,Year,Platform,Category,Sub-Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Loreal Sales Value
5911,Online,2026,Shopee Mall,SKINCARE,SUN CARE,Non-Mass Medical,WINONA,03,68.64,2.0,0.0
5912,Online,2026,Shopee Mall,SKINCARE,SUN CARE,Non-Mass Medical,WINONA,04,95.80,2.0,0.0
5913,Online,2026,Shopee Mall,SKINCARE,SUN CARE,Non-Mass Medical,WINONA,05,95.60,2.0,0.0


In [17]:
sg_ldb_online = pd.concat([ldb_market, ldb_brand], ignore_index=True)

In [18]:
# # Shopee Include Avene Brand But Lazada exclude?
# sg_ldb_online = sg_ldb_online[~((sg_ldb_online['Platform'] == 'Lazada Mall') & (sg_ldb_online['Brand'] == 'AVENE'))]

In [19]:
# Exclude Body Care from Non-Mass Medical?
ldb_market = ldb_market[~((ldb_market['Sub-Category'] == 'Body Care') & (ldb_market['Mass/non-mass (subdivision)'] == 'Non-Mass Medical'))]

### Final Transformation

In [20]:
# Merge Everything
##sg_ldb_final = pd.concat([sg_ldb_offline, sg_ldb_online], ignore_index=True)

# Merge Everything
# sg_ldb_final = pd.concat([sg_ldb_offline], ignore_index=True)
sg_ldb_final = pd.concat([sg_ldb_offline,sg_ldb_online], ignore_index=True)


In [21]:
# Important!! List of Brand to include in the report - New Brands need to be added here too
brand_list = ['SKINCEUTICALS', 'MEDIC MARKET', 'EUCERIN', 'LA ROCHE POSAY', 'VICHY', 'AVENE', 'BIODERMA', 'CUREL', 'URIAGE', 'QV', 'CETAPHIL', 'DR. CI : LABO', 'PHYSIOGEL', 'NEUTROGENA', 'MUSTELA', 'SEBAMED', 'TOPICREM', 'CERAVE', 'EGO', 'AQUAPHOR', 'SKINCEUTICALS', 'DECLEOR', 'SANOFLORE', 'ISDIN', 'LIERAC', 'FILORGA', 'PROACTIV', 'ROC', 'WINONA', 'DR CILABO', 'EMOLIUM', 'PHARMACERIS', 'CAUDALIE', 'NUXE', 'RODAN', 'FIELDS', 'LIBREDERM', 'MANTECORP', 'DR. CI : LABO','CERAVE','DR.G']
print('Total brand_list Brands: ', len(brand_list))

Total brand_list Brands:  41


In [22]:
def map_brand(row):
    if row['Brand'] == 'La Roche-Posay':
        return 'LA ROCHE POSAY'
    elif row['Brand'] == 'SKIN CEUTICALS':
        return 'SKINCEUTICALS'
    elif row['Brand'] == 'DR. CI : LABO':
        return 'DR. CILABO'
    elif row['Brand'] == 'Avène':
        return 'AVENE'
    else:
        return row['Brand']

In [23]:
def map_group(row):
    if row['Brand'] in ['LA ROCHE POSAY', 'VICHY', 'SKINCEUTICALS', 'CERAVE','DR.G']:
        return "L'Oreal"
    elif row['Brand'] == 'MEDIC MARKET':
        return 'Total'
    else:
        return 'Other'

In [24]:
# Category Mapping
def map_category(row):
    if row['Category'] == 'HAIR':
        return 'Haircare'
    elif row['Category'] == 'MAKE UP':
        return 'Makeup'
    elif (row['Category'] == 'SKINCARE') & (row['Sub-Category'] == 'BODY CARE'):
        return 'Bodycare'
    elif (row['Category'] == 'SKINCARE') & (row['Sub-Category'] == 'SUN CARE'):
        return 'Suncare'
    elif (row['Category'] == 'SKINCARE'):
        return 'Facecare'
    else:
        return 'Other'

In [25]:
def map_sellout(row):
    if (row["L'Oreal/Other"] == "L'Oreal") & (row['On/Offline'] == 'Online'):
        return row['Loreal Sales Value']
    else:
        return row['Sales Value']

In [26]:
# Important!! Try to understand the filter and logic here
sg_ldb_final = sg_ldb_final[sg_ldb_final['Brand'].isin(brand_list)]
sg_ldb_final['O+O Brand'] = sg_ldb_final.apply(map_brand, axis=1)
sg_ldb_final["L'Oreal/Other"] = sg_ldb_final.apply(map_group, axis=1)
sg_ldb_final['O+O Category'] = sg_ldb_final.apply(map_category, axis=1)
sg_ldb_final['Sellout'] = sg_ldb_final.apply(map_sellout, axis=1)

# Remove leading zeros and convert to integer
sg_ldb_final['Period'] = sg_ldb_final['Period'].astype(str).str.lstrip('0').astype(int)

In [27]:
# Sort Columns
order = ['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period', 'Sales Value', 'Sales Units']

sg_ldb_final = sg_ldb_final[order]

In [28]:
sg_ldb_final.rename(columns={'Loreal Sales Value': 'ACD 1P'}, inplace=True)
sg_ldb_final ['Year'] = sg_ldb_final ['Year'].astype(int)
sg_ldb_final = sg_ldb_final[sg_ldb_final['Year'] >= 2021]
sg_ldb_final.head(3)

,On/Offline,Year,Platform,Category,Sub-Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units
0,Offline,2023,Nielsen,SKINCARE,,Mass Medical,MEDIC MARKET,4,733640.361,43516.0
1,Offline,2023,Nielsen,SKINCARE,,Mass Medical,MEDIC MARKET,5,819606.816,49404.0
2,Offline,2023,Nielsen,SKINCARE,,Mass Medical,MEDIC MARKET,6,968239.173,55875.0


In [29]:
# Get last month
last_month = datetime.now().replace(day=1) - timedelta(days=1)
month_abbr = last_month.strftime("%b").upper()
year = last_month.year

# Format the output as "MMM YYYY"
filemonth = last_month.strftime("%b %Y").upper()

# Print the result
print(f"{filemonth}")

MAY 2026


In [30]:
if not os.path.exists(f'../Generated Data/O+O/{filemonth}'):
        os.makedirs(f'../Generated Data/O+O/{filemonth}')

In [31]:
sg_ldb_final.to_excel(f'../Generated Data/O+O/{filemonth}/SG LDB {filemonth} O+O.xlsx', index=False)